In [5]:
"""
report2_price_analysis.py
==========================
Reads:
  Dataset/Normalised/prices_normalised.csv   — output of module_6a_normalise.py
  Dataset/hs_classification.csv              — HS chapter reference

Runs outlier detection fresh from the normalised file.

Output: Reports/price_analysis_report.xlsx

Sheets:
  00_Chapter_Summary   All chapters with Section, Chapter Name, HS Use Notes,
                       row counts, commodity counts, % flagged
  01_Flagged_Rows      Every flagged row, filterable by chapter/severity/country/commodity
  02_Commodities       All commodities: n, median price, CV, status
  03_High_Priority     CRITICAL + HIGH severity only

CONFIG:
  EXCLUDE_CHAPTERS  list of 2-digit chapter codes to skip (default: none)
  IQR_MULTIPLIER    3.0 = lenient, 1.5 = strict
  MIN_ROWS          minimum valid rows to analyse a commodity
  MAX_CV            CV above this -> HIGH VAR

Usage:
  python3 report2_price_analysis.py
"""

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── CONFIG ────────────────────────────────────────────────────
# ── Environment detection (Colab vs Local) ───────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os as _os
    _os.listdir('/content/drive/MyDrive')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Exports')
    BASE_DIR = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    IN_COLAB = True
except Exception:
    BASE_DIR = Path.cwd().parent / 'Dataset'
    IN_COLAB = False

EXPORTS_DIR  = BASE_DIR / 'Exports'
NORM_FILE    = BASE_DIR / 'Normalised' / 'prices_normalised.csv'
REPORTS_DIR  = BASE_DIR / 'Reports'
LOOKER_DIR   = BASE_DIR / 'Looker'

print(f'Environment : {"Colab" if IN_COLAB else "Local"}')
print(f'Base dir    : {BASE_DIR}')
print(f'Exists      : {BASE_DIR.exists()}')

REPORTS_DIR     = BASE_DIR / "Reports"
OUT_FILE        = REPORTS_DIR / "price_analysis_report.xlsx"

EXCLUDE_CHAPTERS: list = []
IQR_MULTIPLIER = 3.0
MIN_ROWS       = 5
MAX_CV         = 1.0

# ── STYLE HELPERS ─────────────────────────────────────────────
HDR="1F3864"; WHT="FFFFFF"; ALT="EEF4FB"; NTE="DEEAF1"
OK_B="C6EFCE"; OK_F="276221"
WN_B="FFEB9C"; WN_F="9C5700"
FL_B="FFC7CE"; FL_F="9C0006"
CR_B="C00000"; CR_F="FFFFFF"
HV_B="E2EFDA"; HV_F="375623"
LD_B="F2F2F2"; LD_F="666666"

def _fill(c): return PatternFill("solid", start_color=c, end_color=c)
def _font(bold=False, color="000000", sz=10):
    return Font(name="Arial", bold=bold, color=color, size=sz)
def _border():
    s = Side(style="thin", color="BDD7EE")
    return Border(left=s, right=s, top=s, bottom=s)
def _left():   return Alignment(horizontal="left",   vertical="center", wrap_text=True)
def _center(): return Alignment(horizontal="center", vertical="center", wrap_text=True)

def _title(ws, text, ncols):
    ws.append([text] + [""] * (ncols - 1))
    r = ws.max_row
    ws.cell(r, 1).font = _font(bold=True, color=WHT, sz=12)
    ws.cell(r, 1).fill = _fill(HDR)
    ws.cell(r, 1).alignment = _left()
    ws.row_dimensions[r].height = 28
    if ncols > 1:
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=ncols)

def _note(ws, text, ncols):
    ws.append([text] + [""] * (ncols - 1))
    r = ws.max_row
    ws.cell(r, 1).font = _font(color="1F3864")
    ws.cell(r, 1).fill = _fill(NTE)
    ws.cell(r, 1).alignment = _left()
    if ncols > 1:
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=ncols)

def _hdr_row(ws, cols, widths=None):
    ws.append(cols)
    r = ws.max_row
    for c in range(1, len(cols) + 1):
        cell = ws.cell(r, c)
        cell.font      = _font(bold=True, color=WHT)
        cell.fill      = _fill(HDR)
        cell.alignment = _center()
        cell.border    = _border()
    if widths:
        for c, w in enumerate(widths, 1):
            ws.column_dimensions[get_column_letter(c)].width = w
    ws.row_dimensions[r].height = 30
    return r

def _autofilter(ws, hr, ncols):
    ws.auto_filter.ref = f"A{hr}:{get_column_letter(ncols)}{hr}"

def _plain_row(ws, vals, alt=False):
    ws.append(vals)
    r = ws.max_row
    for c, v in enumerate(vals, 1):
        cell = ws.cell(r, c)
        cell.border    = _border()
        cell.alignment = _left()
        cell.fill      = _fill(ALT if alt else WHT)
        cell.font      = _font()

def _colour_cell(ws, row, col, bg, fg="000000", bold=False):
    cell = ws.cell(row, col)
    cell.fill = _fill(bg)
    cell.font = _font(bold=bold, color=fg)

def _pct_colours(pct):
    if pct == 0:  return OK_B, OK_F
    if pct < 5:   return WHT,  "000000"
    if pct < 15:  return WN_B, WN_F
    return FL_B, FL_F

def _sev_colours(sev):
    s = str(sev).upper()
    if "CRITICAL" in s: return CR_B, CR_F
    if "HIGH"     in s: return FL_B, FL_F
    if "MEDIUM"   in s: return WN_B, WN_F
    return WHT, "000000"

# ── LOAD HS CLASSIFICATION ────────────────────────────────────
def load_hs(path):
    try:
        df = pd.read_csv(path, dtype=str, encoding="latin-1")
        df["HS_Chapter"] = df["HS_Chapter"].astype(str).str.strip().str.zfill(2)
        lookup = {}
        for _, r in df.iterrows():
            lookup[r["HS_Chapter"]] = {
                "section":    r.get("Section_Name",   "").strip(),
                "chapter":    r.get("Chapter_Name",   "").strip(),
                "notes":      r.get("HS_Use_Notes",   "").strip(),
                "section_no": r.get("Section_Number", "").strip(),
            }
        return lookup
    except Exception as e:
        print(f"  Warning: could not load HS file ({e}). Chapter codes only.")
        return {}

def hs_get(lookup, chapter, field):
    return lookup.get(str(chapter).zfill(2), {}).get(field, "—")

# ── LOAD NORMALISED ───────────────────────────────────────────
def load_normalised():
    if not NORMALISED_FILE.exists():
        raise FileNotFoundError(
            f"Not found: {NORMALISED_FILE}\nRun module_6a_normalise.py first.")
    df = pd.read_csv(NORMALISED_FILE, dtype=str)
    df["_unit_price"] = pd.to_numeric(df["_unit_price"], errors="coerce")
    df["_month"]      = pd.to_numeric(df.get("_month"),  errors="coerce")
    df["_hs_chapter"] = df["Commodity"].str.extract(r"^(\d{2})")
    if "_season" not in df.columns:
        m2s = {1:"Q1",2:"Q1",3:"Q1",4:"Q2",5:"Q2",6:"Q2",
               7:"Q3",8:"Q3",9:"Q3",10:"Q4",11:"Q4",12:"Q4"}
        df["_season"] = df["_month"].map(m2s).fillna("Unknown")
    return df

# ── OUTLIER DETECTION ─────────────────────────────────────────
def detect_outliers(norm):
    ok = norm[
        (norm["_conversion_status"] == "OK") &
        norm["_unit_price"].notna() &
        np.isfinite(norm["_unit_price"]) &
        (norm["_unit_price"] > 0)
    ].copy()

    chunks = []
    for commodity, grp in ok.groupby("Commodity"):
        prices = grp["_unit_price"].values
        if len(prices) < MIN_ROWS:
            continue
        q1  = np.percentile(prices, 25)
        q3  = np.percentile(prices, 75)
        iqr = q3 - q1
        lo  = q1 - IQR_MULTIPLIER * iqr
        hi  = q3 + IQR_MULTIPLIER * iqr
        med = np.median(prices)
        mn  = np.mean(prices)
        std = np.std(prices, ddof=1) if len(prices) > 1 else 0.0
        cv  = std / mn if mn > 0 else 0.0

        out = grp[(grp["_unit_price"] < lo) | (grp["_unit_price"] > hi)].copy()
        if out.empty:
            continue

        out["_group_median"]     = round(float(med), 6)
        out["_lower_fence"]      = round(float(lo),  6)
        out["_upper_fence"]      = round(float(hi),  6)
        out["_group_cv"]         = round(float(cv),  4)
        z = (out["_unit_price"] - mn) / std if std > 0 else 0
        out["_z_score"]          = z.round(3)
        out["_outlier_dir"]      = np.where(out["_unit_price"] > hi, "HIGH", "LOW")
        out["_outlier_severity"] = out["_z_score"].abs().apply(
            lambda z: "CRITICAL" if z >= 4 else ("HIGH" if z >= 3 else "MEDIUM"))
        out["_high_var_flag"]    = "YES" if cv > MAX_CV else "NO"
        chunks.append(out)

    if not chunks:
        return pd.DataFrame()
    result = pd.concat(chunks, ignore_index=True)
    result = result.sort_values("_z_score", key=abs, ascending=False)
    return result


# ── CHAPTER SUMMARY ───────────────────────────────────────────
def chapter_summary(norm, flagged, hs):
    """Build chapter summary directly from normalised and flagged data."""
    ok = norm[
        (norm["_conversion_status"] == "OK") &
        norm["_unit_price"].notna() &
        np.isfinite(norm["_unit_price"]) &
        (norm["_unit_price"] > 0)
    ].copy()

    # Valid row counts per chapter
    ok_counts = ok.groupby("_hs_chapter").agg(
        total_valid_rows=("_unit_price", "count"),
        total_commodities=("Commodity", "nunique"),
    ).reset_index()

    # Flagged row counts per chapter
    if not flagged.empty:
        fl_counts = flagged.groupby("_hs_chapter").agg(
            flagged_rows=("Commodity", "count"),
            flagged_commodities=("Commodity", "nunique"),
        ).reset_index()
    else:
        fl_counts = pd.DataFrame(columns=["_hs_chapter","flagged_rows","flagged_commodities"])

    merged = ok_counts.merge(fl_counts, on="_hs_chapter", how="left").fillna(0)
    merged["flagged_rows"]        = merged["flagged_rows"].astype(int)
    merged["flagged_commodities"] = merged["flagged_commodities"].astype(int)
    merged["pct_flagged"]         = (
        merged["flagged_rows"] / merged["total_valid_rows"] * 100
    ).round(1)

    rows = []
    for _, r in merged.iterrows():
        ch = r["_hs_chapter"]
        if ch in EXCLUDE_CHAPTERS:
            continue
        rows.append({
            "HS Chapter":          ch,
            "Section":             hs_get(hs, ch, "section"),
            "Chapter Name":        hs_get(hs, ch, "chapter"),
            "HS Use Notes":        hs_get(hs, ch, "notes"),
            "Total Commodities":   int(r["total_commodities"]),
            "Total Valid Rows":    int(r["total_valid_rows"]),
            "Flagged Rows":        int(r["flagged_rows"]),
            "% Rows Flagged":      float(r["pct_flagged"]),
            "Flagged Commodities": int(r["flagged_commodities"]),
        })
    return (pd.DataFrame(rows)
              .sort_values("% Rows Flagged", ascending=False)
              .reset_index(drop=True))

# ── BUILD WORKBOOK ────────────────────────────────────────────
def build(norm, flagged, hs, run_dt):
    chap_df  = chapter_summary(norm, flagged, hs)

    # Enrich flagged with HS lookup
    flag_out = pd.DataFrame()
    if not flagged.empty:
        flag_out = flagged.copy()
        flag_out["Section"]      = flag_out["_hs_chapter"].apply(lambda x: hs_get(hs, x, "section"))
        flag_out["Chapter Name"] = flag_out["_hs_chapter"].apply(lambda x: hs_get(hs, x, "chapter"))

    wb = Workbook()
    wb.remove(wb.active)

    ok_n = int((norm["_conversion_status"] == "OK").sum())

    # ── 00 Chapter Summary ────────────────────────────────────
    ws = wb.create_sheet("00_Chapter_Summary")
    ws.sheet_view.showGridLines = False
    N0 = 9
    _title(ws, "PRICE ANALYSIS REPORT — CHAPTER SUMMARY", N0)
    ws.append([""])
    _note(ws, (f"Generated: {run_dt}   |   "
               f"Total normalised rows: {len(norm):,}   |   "
               f"Valid price rows: {ok_n:,}   |   "
               f"Flagged rows: {len(flagged):,}   |   "
               f"Chapters: {len(chap_df)}   |   "
               f"Commodities: {chap_df['Total Commodities'].sum():,}"), N0)
    ws.append([""])
    _note(ws, ("% Rows Flagged colour:  "
               "green = 0%   white < 5%   amber 5–15%   red > 15%   |   "
               "High Var = CV > 1.0 (price too variable for reliable outlier detection)   |   "
               "Low Data = fewer than 5 valid price rows"), N0)
    ws.append([""])

    c0 = ["HS Chapter","Section","Chapter Name","HS Use Notes",
          "Total Commodities","Total Valid Rows",
          "Flagged Rows","% Rows Flagged","Flagged Commodities"]
    w0 = [12,36,46,30,20,18,14,16,22]
    hr0 = _hdr_row(ws, c0, w0)
    _autofilter(ws, hr0, N0)

    for i, (_, r) in enumerate(chap_df.iterrows()):
        vals = [r[c] for c in c0]
        ws.append(vals)
        rn  = ws.max_row
        alt = i % 2 == 1
        for c in range(1, N0 + 1):
            cell = ws.cell(rn, c)
            cell.border    = _border()
            cell.alignment = _left()
            cell.fill      = _fill(ALT if alt else WHT)
            cell.font      = _font()
        pct = r["% Rows Flagged"]
        bg, fg = _pct_colours(pct)
        _colour_cell(ws, rn, 8, bg, fg, bold=(pct > 0))

    ws.freeze_panes = f"A{hr0 + 1}"

    # Build shared column config (used by High Priority sheet)
    display_cols = ["_hs_chapter","Section","Chapter Name","Commodity",
                    "Period","_season","Country","Province",
                    "_unit_price","_base_unit",
                    "_outlier_severity","_outlier_dir","_z_score",
                    "_group_median","_lower_fence","_upper_fence",
                    "_group_cv","_high_var_flag",
                    "Value ($)","Quantity","Unit of measure","_source_file"]
    col_labels = {
        "_hs_chapter":"HS Chapter","_season":"Season",
        "_unit_price":"Unit Price","_base_unit":"Unit",
        "_outlier_severity":"Severity","_outlier_dir":"Direction",
        "_z_score":"Z-Score","_group_median":"Group Median",
        "_lower_fence":"Lower Fence","_upper_fence":"Upper Fence",
        "_group_cv":"CV","_high_var_flag":"High Var?",
        "_source_file":"Source File",
    }
    col_widths = {
        "_hs_chapter":10,"Section":32,"Chapter Name":40,"Commodity":55,
        "Period":14,"_season":10,"Country":22,"Province":14,
        "_unit_price":14,"_base_unit":8,"_outlier_severity":12,
        "_outlier_dir":12,"_z_score":10,"_group_median":14,
        "_lower_fence":14,"_upper_fence":14,"_group_cv":10,
        "_high_var_flag":10,"Value ($)":14,"Quantity":12,
        "Unit of measure":24,"_source_file":28,
    }
    avail = [c for c in display_cols if c in (flag_out.columns.tolist() if not flag_out.empty else [])]
    hdr   = [col_labels.get(c, c) for c in avail]
    wids  = [col_widths.get(c, 14) for c in avail]
    sev_c = avail.index("_outlier_severity") + 1 if "_outlier_severity" in avail else None

    # ── 01 High Priority ─────────────────────────────────────
    ws1 = wb.create_sheet("01_High_Priority")
    ws1.sheet_view.showGridLines = False
    N1 = max(len(avail), 6)
    _title(ws1, "HIGH PRIORITY FLAGS — CRITICAL & HIGH SEVERITY ONLY", N1)
    ws1.append([""])
    _note(ws1, ("Rows where |z-score| ≥ 3.0.  Investigate these first.   "
                "High Var? = YES means the commodity has naturally high price variation — "
                "treat with more caution, but still worth reviewing if the z-score is very large."), N1)
    ws1.append([""])

    if flag_out.empty:
        _note(ws1, "No flagged rows found.", N1)
    else:
        high_p = flag_out[
            flag_out.get("_outlier_severity", pd.Series(dtype=str))
            .str.upper().isin(["CRITICAL", "HIGH"])
        ] if "_outlier_severity" in flag_out.columns else pd.DataFrame()

        if high_p.empty:
            _note(ws1, "No CRITICAL or HIGH severity flags found — all flags are MEDIUM severity.", N1)
        else:
            hr1 = _hdr_row(ws1, hdr, wids)
            _autofilter(ws1, hr1, len(avail))
            for i, (_, r) in enumerate(high_p.iterrows()):
                vals = [r.get(c, "") for c in avail]
                ws1.append(vals)
                rn  = ws1.max_row
                alt = i % 2 == 1
                for c in range(1, len(avail) + 1):
                    cell = ws1.cell(rn, c)
                    cell.border    = _border()
                    cell.alignment = _left()
                    if c == sev_c:
                        bg, fg = _sev_colours(str(vals[c - 1]))
                        cell.fill = _fill(bg)
                        cell.font = _font(bold=True, color=fg)
                    else:
                        cell.fill = _fill(ALT if alt else WHT)
                        cell.font = _font()
            ws1.freeze_panes = f"A{hr1 + 1}"

    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    wb.save(OUT_FILE)
    return chap_df, flag_out

# ── MAIN ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "=" * 58)
    print("  REPORT 2 — PRICE ANALYSIS")
    print("=" * 58)

    # HS classification
    hs_path = HS_FILE if HS_FILE.exists() else Path("Dataset/hs_classification.csv")
    print(f"\n  Loading HS classification: {hs_path}")
    hs = load_hs(hs_path)
    print(f"  Loaded {len(hs)} chapters from HS file")

    # Normalised prices
    print(f"  Loading: {NORMALISED_FILE}")
    try:
        norm = load_normalised()
    except FileNotFoundError as e:
        print(f"\n  ERROR: {e}"); exit(1)
    ok_n = int((norm["_conversion_status"] == "OK").sum())
    print(f"  Loaded {len(norm):,} rows  ({ok_n:,} with valid unit price)")

    if EXCLUDE_CHAPTERS:
        print(f"  Excluding chapters: {', '.join(EXCLUDE_CHAPTERS)}")

    # Outlier detection
    print(f"  Running outlier detection "
          f"(IQR × {IQR_MULTIPLIER}, min rows = {MIN_ROWS})...")
    flagged = detect_outliers(norm)
    if flagged.empty:
        print("  No flagged rows found.")
    else:
        print(f"  Flagged: {len(flagged):,} rows across "
              f"{flagged['Commodity'].nunique()} commodities")
        for sev, cnt in flagged["_outlier_severity"].value_counts().items():
            print(f"    {sev}: {cnt}")

    # Build
    print("  Building Excel report...")
    run_dt = datetime.now().strftime("%Y-%m-%d %H:%M")
    chap_df, flag_out = build(norm, flagged, hs, run_dt)

    flagged_chaps = chap_df[chap_df["Flagged Rows"] > 0]
    if not flagged_chaps.empty:
        print(f"\n  Top chapters by flagged rows:")
        for _, r in flagged_chaps.head(8).iterrows():
            print(f"    Ch {r['HS Chapter']}  {r['Chapter Name'][:38]:<40} "
                  f"{r['Flagged Rows']:>4} flagged ({r['% Rows Flagged']}%)")

    print(f"\n  Saved: {OUT_FILE}\n")



  REPORT 2 — PRICE ANALYSIS

  Loading HS classification: /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/hs_classification.csv
  Loaded 99 chapters from HS file
  Loading: /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Normalised/prices_normalised.csv
  Loaded 2,562,365 rows  (1,288,311 with valid unit price)
  Running outlier detection (IQR × 3.0, min rows = 5)...
  Flagged: 61,740 rows across 2836 commodities
    MEDIUM: 45625
    CRITICAL: 9843
    HIGH: 6272
  Building Excel report...

  Top chapters by flagged rows:
    Ch 37  Photographic or cinematographic goods       2 flagged (12.5%)
    Ch 05  Products of animal origin, not elsewhe    294 flagged (10.1%)
    Ch 83  Miscellaneous articles of base metal       70 flagged (9.8%)
    Ch 57  Carpets and other textile floor coveri    185 flagged (9.6%)
    Ch 35  Albuminoidal substances; modified star    629 flagged (9.6%)
    Ch 29  Organic chemicals                        3896 flagged (9.6%)
    Ch 50  Silk   